# Rank Uncertain Designs

This example demonstrates how to rank designs whose objectives are uncertain with the `tatc.tradespace` module: the probability that each design is Pareto-efficient given the uncertainty of its replicates, replicates added only where they settle it, new replicates that confirm the selected designs, and statistics other than means that state how reliably a design performs.

It continues the example `SearchTradeSpace`, whose design space and study it imports from `trade_space_example.py` (beside the notebooks). The evaluations of every design are taken from the study's store if `SearchTradeSpace` was run before (or are evaluated again, in a few minutes). Its objectives are the numbers of satellites and of planes and the share of points sufficiently sampled (at least four times a day), estimated from five replicates (samples of points) of each design.

In [1]:
import pandas as pd

from trade_space_example import OBJECTIVES, describe, designs, study

# the evaluations of every design
results = study.run(designs)
print(f"{len(results)} evaluations: {len(designs)} designs x {study.replicates} replicates")

5480 evaluations: 1096 designs x 5 replicates


## Rank Designs with the Uncertainty of Replicates

The means of five replicates are uncertain, and so is the front of the means: a design just behind another in the means may be ahead of it in other samples of points. `pareto_probability` estimates the probability that each design is efficient by a bootstrap: it resamples the replicates (the same replicates for every design, which keeps the correlation between designs of common random numbers) and counts how often each design is on the front of the resampled means.

In [2]:
from tatc.tradespace import pareto_probability

probability = pareto_probability(results, OBJECTIVES, n_resamples=1000)
probability["description"] = probability.apply(describe, axis=1)
candidates = probability[probability.pareto_probability >= 0.1].sort_values(["satellites", "planes", "sufficient"])
candidates[["description", "satellites", "planes", "sufficient", "pareto", "pareto_probability"]].round(2)

,description,satellites,planes,sufficient,pareto,pareto_probability
1017,"800 km, 60 deg FOR, Walker 75 deg: 1 x 1 (spac...",1.0,1.0,2.8,False,0.38
776,"700 km, 60 deg FOR, SSO: 1 at 12:00",1.0,1.0,3.0,False,0.20
1048,"800 km, 60 deg FOR, SSO: 1 at 09:00",1.0,1.0,3.0,False,0.12
1050,"800 km, 60 deg FOR, SSO: 1 at 12:00",1.0,1.0,3.0,False,0.10
1052,"800 km, 60 deg FOR, SSO: 1 at 15:00",1.0,1.0,3.0,False,0.12
1054,"800 km, 60 deg FOR, SSO: 1 at 18:00",1.0,1.0,3.2,True,0.54
960,"800 km, 60 deg FOR, Walker 45 deg: 1 x 2 (spac...",2.0,1.0,16.6,True,0.98
963,"800 km, 60 deg FOR, Walker 45 deg: 2 x 1 (spac...",2.0,2.0,30.2,True,1.00
961,"800 km, 60 deg FOR, Walker 45 deg: 1 x 3 (spac...",3.0,1.0,38.0,True,1.00
972,"800 km, 60 deg FOR, Walker 45 deg: 3 x 1 (spac...",3.0,3.0,37.6,False,0.32


Four of the seven efficient designs are efficient in every resample: two satellites in two planes, three or four satellites in one plane, and six in two planes; two satellites in one plane (efficient in 98% of the resamples) and four in two planes (92%) nearly so. The single satellite is not settled by five replicates: one in a sun-synchronous orbit crossing the equator at 18:00 (efficient in 54% of the resamples) samples about as many points as one in a Walker orbit inclined 75 degrees (38%) or crossing at other times (10 to 20%). Likewise, three satellites in three planes (29% and 32%) sample about as many points as three in one plane, and six satellites in two planes with another relative spacing (32%) as many as the efficient design. Designs that are efficient with a probability between 0 and 1 are equally good candidates, given the uncertainty of the estimates: more replicates (samples of points) would separate them, or a tolerance of the share (see `pareto_front`) would treat them as ties.

## Refine Uncertain Designs

Five replicates settle the efficiency of most designs, but not of all. More replicates of every design would settle them, at the cost of evaluating every design again for each. `refine` instead adds replicates only to the designs whose efficiency is uncertain (efficient in more than 5% and fewer than 95% of the resamples), five at a time, and estimates their probabilities again, until none is uncertain (or a design has 40 replicates). The added replicates are the next ones after each design's own, which keeps common random numbers between the designs refined together.

Some designs differ by less than any number of replicates could separate, such as those that sample nearly every point: a difference of a tenth of a percentage point does not matter to the trade. A tolerance of 1 percentage point of the share, part of the criteria by which designs are judged (`Criteria`), treats designs within it of the front as efficient (see `pareto_front`), so that refinement does not spend replicates on differences that do not matter. A design about one tolerance below the front stays uncertain instead: more replicates settle which side of the tolerance it falls on only slowly, though it does not matter. `refine` stops refining such a design once the standard error of its share is less than half the tolerance (`precision`): it is then `indifferent`. A `budget` would limit the evaluations, refining the least settled designs first.

In [3]:
from tatc.tradespace import Criteria, refine

TOLERANCE = {"sufficient": 1.0}  # percentage points of the share that do not matter
CRITERIA = Criteria(OBJECTIVES, tolerance=TOLERANCE)
refinement = refine(study, CRITERIA, results, step=5, max_replicates=40, precision=0.5, n_resamples=1000)
added = len(refinement.results) - len(results)
print(f"{added} evaluations added, {added / (len(designs) * 35):.1%} of those of 40 replicates of every design")
refinement.history

215 evaluations added, 0.6% of those of 40 replicates of every design


,round,uncertain,indifferent,refined,evaluations
0,0,31,2,29,145
1,1,16,11,5,25
2,2,14,11,3,15
3,3,14,12,2,10
4,4,14,13,1,5
5,5,14,13,1,5
6,6,14,12,2,10
7,7,13,12,0,0


In [4]:
refined = refinement.summary
refined["description"] = refined.apply(describe, axis=1)
refined[refined.pareto_probability >= 0.1].sort_values(["satellites", "planes", "sufficient"])[
    ["description", "satellites", "planes", "sufficient", "replicates", "pareto", "pareto_probability", "indifferent"]
].round(2)

,description,satellites,planes,sufficient,replicates,pareto,pareto_probability,indifferent
500,"600 km, 60 deg FOR, SSO: 1 at 09:00",1.0,1.0,1.80,10,False,0.12,True
230,"500 km, 60 deg FOR, SSO: 1 at 15:00",1.0,1.0,1.90,10,False,0.14,True
498,"600 km, 60 deg FOR, SSO: 1 at 06:00",1.0,1.0,1.90,10,False,0.19,True
504,"600 km, 60 deg FOR, SSO: 1 at 15:00",1.0,1.0,2.10,10,False,0.18,True
502,"600 km, 60 deg FOR, SSO: 1 at 12:00",1.0,1.0,2.20,10,True,0.40,True
506,"600 km, 60 deg FOR, SSO: 1 at 18:00",1.0,1.0,2.20,10,True,0.32,True
1017,"800 km, 60 deg FOR, Walker 75 deg: 1 x 1 (spac...",1.0,1.0,2.40,10,True,0.57,True
774,"700 km, 60 deg FOR, SSO: 1 at 09:00",1.0,1.0,2.50,10,True,0.76,True
780,"700 km, 60 deg FOR, SSO: 1 at 18:00",1.0,1.0,2.50,10,True,0.79,True
778,"700 km, 60 deg FOR, SSO: 1 at 15:00",1.0,1.0,2.53,15,True,0.70,True


Refinement adds 215 evaluations, 0.6% of those that 40 replicates of every design would take, to the few designs whose efficiency is uncertain. With the tolerance, the designs that sample nearly every point (six to nine satellites, about 98%) are efficient candidates, with probabilities of 0.97 or more, rather than uncertain: they differ by less than matters. The designs that remain uncertain lie about one tolerance below the front, such as single satellites that sample 1.8 to 2.5% of the points against the best single satellite's 3.2%, or nine satellites that sample 96.8% against 97.8%: whether they are within the tolerance of the front is a question that more replicates settle only slowly, and one that does not change the trade. They are indifferent once their shares are known to within half the tolerance (a standard error of 0.5 percentage points), after 10 or 15 replicates of the single satellites (and the first five of the nine satellites); without a precision (`precision=None`), refinement would refine them to the limit of 40 replicates, for 555 evaluations. The shares of three satellites vary more between replicates: three satellites in three planes that sample 38.5% are indifferent after 25 replicates, and those that sample 37.0% remain uncertain with 40. An absolute tolerance suits a share near saturation better than one near zero; a tolerance proportional to the objective would treat both alike.

## Confirm the Front

The efficient designs were selected for their means over their replicates, so they are selected in part for the luck of those replicates: among a thousand designs, those that look best are likely to have drawn favorable samples of points (the winner's curse). `confirm` evaluates the selected designs (by default, the front of the results it is given) with new replicates, which played no part in their selection (the same for every design, which keeps common random numbers), and finds the front among them by these replicates. The difference between the means of the replicates that selected the designs and those of the new ones measures the optimism of the selection.

In [5]:
from tatc.tradespace import confirm

# the front of the refined results, with 20 new replicates
confirmation = confirm(study, CRITERIA, refinement.results, replicates=20)
confirmed = confirmation.summary
confirmed["description"] = confirmed.apply(describe, axis=1)
print(f"optimism of the selection: {confirmation.optimism()['sufficient']:.2f} percentage points")
confirmed.sort_values(["satellites", "planes", "sufficient"])[
    ["description", "satellites", "planes", "sufficient selected", "sufficient", "replicates", "pareto"]
].round(2)

optimism of the selection: -0.02 percentage points


,description,satellites,planes,sufficient selected,sufficient,replicates,pareto
4,"600 km, 60 deg FOR, SSO: 1 at 18:00",1.0,1.0,2.20,2.25,20,False
3,"600 km, 60 deg FOR, SSO: 1 at 12:00",1.0,1.0,2.20,2.35,20,False
23,"800 km, 60 deg FOR, Walker 75 deg: 1 x 1 (spac...",1.0,1.0,2.40,2.60,20,False
15,"700 km, 60 deg FOR, SSO: 1 at 15:00",1.0,1.0,2.53,2.85,20,True
16,"700 km, 60 deg FOR, SSO: 1 at 18:00",1.0,1.0,2.50,2.85,20,True
13,"700 km, 60 deg FOR, SSO: 1 at 09:00",1.0,1.0,2.50,2.90,20,True
14,"700 km, 60 deg FOR, SSO: 1 at 12:00",1.0,1.0,2.80,2.95,20,True
12,"700 km, 60 deg FOR, SSO: 1 at 06:00",1.0,1.0,2.90,3.45,20,True
35,"800 km, 60 deg FOR, SSO: 1 at 09:00",1.0,1.0,3.20,3.50,20,True
38,"800 km, 60 deg FOR, SSO: 1 at 18:00",1.0,1.0,3.10,3.50,20,True


The selection is optimistic where it chose the best of nearly equal designs. The designs of two to four satellites sample 0.5 to 3.2 percentage points fewer with the new replicates: four satellites in two planes 66.4% rather than 69.6%, and in one plane 66.5% rather than 68.2%. The design of three satellites in three planes that refinement left as good as three in one plane samples 35.6% with the new replicates (rather than 38.5%) and is no longer efficient. The single satellites and the designs of six to nine satellites instead sample slightly more points with the new replicates (by up to 0.9 and 0.5 points), as the new samples of points happen to be sampled more easily at both ends of the front: common random numbers share this difference between samples among the designs, and it is not an effect of the selection. The mean optimism over the designs (-0.02 points) mixes the two, so compare the designs one by one. All but four of the confirmed designs remain efficient within the tolerance (three single satellites that refinement left indifferent, about a tolerance below the front, and the three satellites in three planes): the shape of the front is confirmed, but its middle is up to 3 points less good than it looked.

## Require Reliable Sampling

A mean share of points sufficiently sampled does not say how reliably a design achieves it: the share differs between samples of points. Statistics over the replicates other than the mean state how a design is judged, and requirements on them are output constraints. Here, the objective is the share that a design exceeds in 80% of the samples (its 0.2 quantile, `Quantile`), and a design is required to sample at least half of the points sufficiently in at least 80% of the samples (the probability of meeting a threshold, `Probability`, with a `Requirement`). `Criteria` collect the objectives, the statistics they name, and the requirements: `aggregate` gives the statistics and the violations of the requirements, and `pareto_probability` the probability that each design is efficient among those that meet them.

In [6]:
from tatc.tradespace import Probability, Quantile, Requirement

RELIABLE = Criteria(
    {"satellites": "min", "planes": "min", "sufficient p20": "max"},
    statistics={
        "sufficient p20": Quantile("sufficient", 0.2),
        "reliability": Probability("sufficient", ">=", 50),
    },
    requirements=[Requirement("reliability", ">=", 0.8)],
)

reliable = pareto_probability(refinement.results, RELIABLE, n_resamples=1000)
reliable["description"] = reliable.apply(describe, axis=1)
print(f"{reliable.meets_requirements.sum()} of {len(reliable)} designs meet the requirement")
reliable[reliable.pareto_probability >= 0.1].sort_values(["satellites", "planes"])[
    ["description", "satellites", "planes", "sufficient p20", "reliability", "replicates", "pareto_probability", "requirements_probability"]
].round(2)

209 of 1096 designs meet the requirement


,description,satellites,planes,sufficient p20,reliability,replicates,pareto_probability,requirements_probability
962,"800 km, 60 deg FOR, Walker 45 deg: 1 x 4 (spac...",4.0,1.0,65.0,1.0,5,1.00,1.0
965,"800 km, 60 deg FOR, Walker 45 deg: 2 x 2 (spac...",4.0,2.0,68.0,1.0,5,0.94,1.0
1025,"800 km, 60 deg FOR, Walker 75 deg: 2 x 3 (spac...",6.0,2.0,97.8,1.0,5,1.00,1.0
1026,"800 km, 60 deg FOR, Walker 75 deg: 2 x 3 (spac...",6.0,2.0,97.0,1.0,5,0.50,1.0


209 of the 1,096 designs meet the requirement of sampling half of the points sufficiently in at least 80% of the samples, and none of them has fewer than four satellites: the best three satellites sample 38% of the points on average. Among those that meet it, four satellites in one plane (inclined 45 degrees) are efficient for sure, sampling at least 65% of the points in 80% of the samples, and so are six satellites in two planes inclined 75 degrees (at least 98%). Four satellites in two planes sample more (at least 68%) and are efficient in 94% of the resamples, and six satellites in two planes with another relative spacing in 50%: these are the alternatives to weigh with other criteria, such as the cost of launching more planes.

## Summary

* The front of the means of the replicates is uncertain: `pareto_probability` finds the designs whose efficiency the replicates settle, and those that are equally good candidates.
* `refine` adds replicates only to the designs whose efficiency is uncertain, with a tolerance of differences that do not matter (part of the `Criteria`), and stops refining designs whose objectives are known to within a share of the tolerance.
* `confirm` evaluates the selected designs (by default, the front of the results) with new replicates, which removes the optimism of selecting the best of many nearly equal designs.
* Statistics other than means (such as a `Quantile` or the `Probability` of meeting a threshold) state how reliably a design performs, and `Requirement`s on them are output constraints.
* Each of these steps takes the study, the criteria, and the results of the step before (or a search's, see `SearchTradeSpace`), and returns its own results and a summary of its designs.